# 6 · Validate the eval judge (Nova Pro)

The pairwise verdict flipped with the judge, and Nova preferred the *untuned base* over SFT — a sanity-check failure that smells like length bias. Before trusting any win-rate we do two things here:

1. **Length-bias probe** (judge-only, runs now): does Nova prefer the longer summary?
2. **Human scoring sheet** (build now, score later): 30 pairs for a human to label, so we can measure human-vs-judge agreement next step.

> Dependency-light: needs `boto3`, `numpy`, and `../src/judge_reward.py` (stdlib + boto3 only — no transformers/datasets). Reads `results/samples.json` from `04_evaluate`.

In [ ]:
import json, csv, random, sys, os
sys.path.insert(0, '../src')
import numpy as np
import judge_reward as J

cfg = json.load(open('summ_config.json'))
SAMPLES = 'results/samples.json'          # downloaded by 04_evaluate
samples = json.load(open(SAMPLES))
models = list(samples[0]['generations'].keys())
print('samples:', len(samples), '| models:', models)
judge = J.JudgeClient(backend='bedrock', model=cfg['eval_judge_model'],
                      region=cfg['region'], max_concurrency=4)
print('eval judge:', cfg['eval_judge_model'])

## 1 · Length-bias probe (judge-only)
For each post we ask the judge to pick between two models' summaries (order randomized), then check whether it chose the **longer** one. If it picks longer ≫50% of the time — especially on the big-gap base-vs-SFT pairs — it has a verbosity bias and its verdicts are suspect.

In [ ]:
def length_bias(m1, m2, label, seed=0):
    rnd = random.Random(seed)
    picked_longer = ties = n = 0
    wlen, llen = [], []
    for rec in samples:
        a, b = rec['generations'][m1], rec['generations'][m2]
        swap = rnd.random() < 0.5                      # randomize A/B presentation
        A, B = (b, a) if swap else (a, b)
        v = judge.compare(rec['post'], A, B)
        if v == 'tie':
            ties += 1; continue
        winner, loser = (A, B) if v == 'A' else (B, A)
        lw, ll = len(winner.split()), len(loser.split())
        wlen.append(lw); llen.append(ll)
        picked_longer += int(lw > ll); n += 1
    frac = picked_longer / max(n, 1)
    print(f'{label}: judge picked the LONGER summary {picked_longer}/{n} = {frac:.0%}  (ties {ties})')
    print(f'   mean winner length={np.mean(wlen):.1f}w  mean loser length={np.mean(llen):.1f}w')
    return frac

if 'base' in models:
    length_bias('base', 'SFT', 'base vs SFT   (big length gap)')
length_bias('SFT', 'SFT+GRPO', 'SFT vs GRPO  (similar length)')
print('\nRule of thumb: >~65% "picked longer" = verbosity bias -> distrust this judge\'s pairwise verdicts')
print('(incl. base>SFT and GRPO>SFT). A clean judge should also prefer SFT over the untuned base.')

## 2 · Build the 30-item human scoring sheet
Writes `human_labeling_sheet.csv` (anonymized A/B so the labeler isn't biased) + a private `human_labeling_key.json`. Give the CSV to a colleague; keep the key. Compares **SFT vs SFT+GRPO** (the decision).

In [ ]:
PAIR = ('SFT', 'SFT+GRPO')
SHEET_N = 50          # how many pairs to put in the sheet (use all or part of samples.json)
N = min(SHEET_N, len(samples))
if len(samples) < SHEET_N:
    print(f'NOTE: only {len(samples)} samples available; sheet will have {N}. Re-run 04_evaluate (saves 50) for more.')
rnd = random.Random(123)
rows, key = [], []
for i, rec in enumerate(samples[:N]):
    a_model, b_model = PAIR if rnd.random() < 0.5 else PAIR[::-1]
    rows.append({'id': i, 'post': ' '.join(rec['post'].split()),  # FULL post, flattened to one tidy cell (no truncation — labelers must see what the model saw)
                 'summary_A': rec['generations'][a_model],
                 'summary_B': rec['generations'][b_model],
                 'better(A/B/tie)': '', 'faithful_A(1-5)': '', 'faithful_B(1-5)': ''})
    key.append({'id': i, 'A_model': a_model, 'B_model': b_model})
with open('human_labeling_sheet.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
json.dump(key, open('human_labeling_key.json', 'w'), indent=2)
print(f'wrote human_labeling_sheet.csv ({N} pairs) + human_labeling_key.json')
print('Labeler fills: better (A/B/tie) + faithfulness 1-5 for each. Do NOT share the key file.')

## 3 · (RUN LATER) Human-vs-judge agreement
After the sheet is filled in, this measures whether Nova agrees with the human: pairwise agreement, and **ordinal** faithfulness agreement (quadratic-weighted kappa — numpy, no deps). Usable ≥ ~0.67.

In [ ]:
def _qwk(a, b, lo=1, hi=5):
    a, b = np.asarray(a, int), np.asarray(b, int); n = hi - lo + 1
    O = np.zeros((n, n))
    for x, y in zip(a, b): O[x-lo, y-lo] += 1
    w = np.array([[((i-j)**2)/((n-1)**2) for j in range(n)] for i in range(n)])
    E = np.outer(O.sum(1), O.sum(0)) / max(O.sum(), 1)
    d = (w*E).sum(); return 1 - (w*O).sum()/d if d > 0 else 1.0

key = {k['id']: k for k in json.load(open('human_labeling_key.json'))}
rows = [r for r in csv.DictReader(open('human_labeling_sheet.csv')) if r['better(A/B/tie)'].strip()]
print(f'labeled rows: {len(rows)}')

# pairwise agreement (human vs judge), judge sees the SAME A/B order as the sheet
pw_agree = 0; hf, jf = [], []
for r in rows:
    i = int(r['id']); rec = samples[i]
    a_model, b_model = key[i]['A_model'], key[i]['B_model']
    A, B = rec['generations'][a_model], rec['generations'][b_model]
    jv = judge.compare(rec['post'], A, B)                 # 'A'/'B'/'tie'
    hv = {'a': 'A', 'b': 'B', 'tie': 'tie'}.get(r['better(A/B/tie)'].strip().lower(), 'tie')
    pw_agree += int(jv == hv)
    for side, model in (('A', a_model), ('B', b_model)):
        h = r[f'faithful_{side}(1-5)'].strip()
        if h:
            hf.append(int(float(h)))
            jf.append(int(round(judge.score(rec['post'], rec['generations'][model])['faithfulness']*4 + 1)))
print(f'pairwise agreement (human vs {judge.model}): {pw_agree}/{len(rows)} = {pw_agree/max(len(rows),1):.0%}')
if hf:
    print(f'faithfulness ordinal agreement (QWK): {_qwk(hf, jf):.3f}   (usable >= ~0.67)')
print('\nIf agreement is low, this eval judge is not trustworthy here -> use a judge that passes,')
print('and remember Sonnet already passed the base-vs-SFT sanity check.')